> 在线讲义：[第 53 章 · 多视几何：对极约束、本质与基础矩阵、单应、三角化、RANSAC、DUSt3R](https://wikieden.github.io/math-and-eval-stats/lessons/53-multiview-geometry.html) · 本书《数学与实验统计》

# 第 53 章 · 多视几何：对极约束、本质与基础矩阵、单应、三角化、RANSAC、DUSt3R

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/53-multiview-geometry.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 对极约束 | x₂ᵀEx₁ | `epipolar_residuals()` | 全 0 |
| 第二步 基础矩阵 | K⁻ᵀEK⁻¹ | `fundamental()` | 秩 2 |
| 第三步 单应 | R + tnᵀ/d | `plane_homography(n, d)` | 和真投影一致 |
| 第四步 三角化 | DLT | `triangulate(x1, x2)` | 误差 < 1e-6 |
| 第五步 RANSAC | log 公式 | `ransac_iters(0.5, 8)` | 1177 |

例子：两台相机（第二台在第一台右边 0.5 m、绕竖直轴转 5°），看同一组 3D 点；本质矩阵 E = [t]× R 的对极约束；
一个平面上的点之间的单应；两视图三角化（DLT）；RANSAC 的迭代次数公式。
里程碑 ME-M32 的“八点法估本质矩阵 + RANSAC（带外点），对照 OpenCV”留给自己做；这里 E 直接由已知的 R、t 构造。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
# 下面几段来自 ch52_camera_geometry.py（第 52 章）
import numpy as np
K = np.array([[500.0, 0, 320], [0, 500.0, 240], [0, 0, 1]])

**函数 `skew`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def skew(t: np.ndarray) -> np.ndarray:
    return np.array([[0, -t[2], t[1]], [t[2], 0, -t[0]], [-t[1], t[0], 0]])

**函数 `two_cams`**：第一台相机在原点；第二台：p2 = R p1 + t。

In [ ]:
def two_cams():
    """第一台相机在原点；第二台：p2 = R p1 + t。"""
    th = np.radians(5)
    r = np.array([[np.cos(th), 0, np.sin(th)], [0, 1, 0], [-np.sin(th), 0, np.cos(th)]])
    t = np.array([-0.5, 0.0, 0.0])
    return r, t

In [ ]:
PTS = np.array([[0.0, 0.0, 4.0], [1.0, -0.5, 5.0], [-1.0, 0.3, 6.0], [0.5, 0.8, 3.0], [-0.4, -0.6, 8.0]])

**函数 `normalized`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def normalized(p: np.ndarray) -> np.ndarray:
    return p / p[2]

**函数 `epipolar_residuals`**：x2ᵀ E x1（归一化坐标）：真对应点应该为 0；返回每个点的值。

In [ ]:
def epipolar_residuals(noise: float = 0.0, seed: int = 0) -> np.ndarray:
    """x2ᵀ E x1（归一化坐标）：真对应点应该为 0；返回每个点的值。"""
    r, t = two_cams()
    e = skew(t) @ r
    rng = np.random.default_rng(seed)
    out = []
    for p in PTS:
        x1 = normalized(p)
        x2 = normalized(r @ p + t)
        x2 = x2 + np.r_[noise * rng.standard_normal(2), 0]
        out.append(float(x2 @ e @ x1))
    return np.array(out)

**函数 `fundamental`**：像素坐标下 F = K⁻ᵀ E K⁻¹，约束 u2ᵀ F u1 = 0；秩 2。

In [ ]:
def fundamental() -> np.ndarray:
    """像素坐标下 F = K⁻ᵀ E K⁻¹，约束 u2ᵀ F u1 = 0；秩 2。"""
    r, t = two_cams()
    ki = np.linalg.inv(K)
    return ki.T @ skew(t) @ r @ ki

**函数 `plane_homography`**：平面 nᵀp = d 上的点在两视图之间满足 x2 ∝ H x1，H = R + t nᵀ/d（归一化坐标）。

In [ ]:
def plane_homography(n: np.ndarray, d: float) -> np.ndarray:
    """平面 nᵀp = d 上的点在两视图之间满足 x2 ∝ H x1，H = R + t nᵀ/d（归一化坐标）。"""
    r, t = two_cams()
    return r + np.outer(t, n) / d

**函数 `triangulate`**：DLT 三角化：每个视图给两条线性方程 x × (P X) = 0，SVD 取最小奇异向量。

In [ ]:
def triangulate(x1: np.ndarray, x2: np.ndarray) -> np.ndarray:
    """DLT 三角化：每个视图给两条线性方程 x × (P X) = 0，SVD 取最小奇异向量。"""
    r, t = two_cams()
    p1 = np.hstack([np.eye(3), np.zeros((3, 1))])
    p2 = np.hstack([r, t[:, None]])
    a = np.vstack([x1[0] * p1[2] - p1[0], x1[1] * p1[2] - p1[1], x2[0] * p2[2] - p2[0], x2[1] * p2[2] - p2[1]])
    x = np.linalg.svd(a)[2][-1]
    return x[:3] / x[3]

**函数 `ransac_iters`**：要以概率 p 至少抽到一次全是内点的样本（每次抽 s 个）：N = log(1 − p) / log(1 − w^s)。

In [ ]:
def ransac_iters(inlier: float, s: int, p: float = 0.99) -> int:
    """要以概率 p 至少抽到一次全是内点的样本（每次抽 s 个）：N = log(1 − p) / log(1 − w^s)。"""
    return int(np.ceil(np.log(1 - p) / np.log(1 - inlier ** s)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 对极约束：x2ᵀ E x1 = 0，E = [t]× R")
print(f"  5 个真对应点：{np.round(epipolar_residuals(), 12).tolist()}")
print(f"  第二个视图加 0.002 的噪声：{np.round(epipolar_residuals(0.002), 5).tolist()}")
r, t = two_cams()
e = skew(t) @ r
print(f"  E 的奇异值 {np.round(np.linalg.svd(e, compute_uv=False), 4).tolist()}（两个相等、一个为 0）")

In [ ]:
print("\n第二步 基础矩阵：换成像素坐标")
f = fundamental()
u1 = K @ normalized(PTS[1])
u2 = K @ normalized(r @ PTS[1] + t)
print(f"  u2ᵀ F u1 = {u2 @ f @ u1:.2e}；F 的秩 {np.linalg.matrix_rank(f, tol=1e-10)}")
line = f @ u1
print(f"  对极线 l2 = F u1 = {np.round(line / np.linalg.norm(line[:2]), 4).tolist()}；u2 到它的距离 {abs(u2 @ line) / np.linalg.norm(line[:2]):.2e} 像素")

In [ ]:
print("\n第三步 单应：平面上的点一一对应")
n, d = np.array([0.0, 0.0, 1.0]), 6.0
h = plane_homography(n, d)
p = np.array([0.7, -0.4, 6.0])
x2 = normalized(h @ normalized(p))
print(f"  平面 Z = 6 上的点 {p.tolist()}：H x1 → {np.round(x2, 5).tolist()}，真值 {np.round(normalized(r @ p + t), 5).tolist()}")

In [ ]:
print("\n第四步 三角化")
for pt in PTS[:3]:
    x1, x2 = normalized(pt), normalized(r @ pt + t)
    print(f"  真值 {pt.tolist()} → 三角化 {np.round(triangulate(x1, x2), 6).tolist()}")

In [ ]:
print("\n第五步 RANSAC 要抽多少次")
for w, s in ((0.5, 8), (0.5, 5), (0.8, 8), (0.5, 3)):
    print(f"  内点率 {w}、每次抽 {s} 个、成功率 99%：{ransac_iters(w, s)} 次")

## 3. 代码里才看得到的

1. **E 由已知的 R、t 直接构造**：代码没有从对应点估计 E；用八点法 + RANSAC（带外点）估计并和 OpenCV 对数，留给自己做（里程碑 ME-M32）。
2. **三角化用归一化坐标**：投影矩阵是 [I | 0] 和 [R | t]，不带 K；带像素坐标时要换成 K[R | t]。
3. **RANSAC 次数向上取整**：公式给出的是实数，`np.ceil` 后才是要跑的次数；实际实现还会根据当前最好模型的内点率自适应地减少次数。

## 4. 自己改一改

1. 把第二台相机的平移改成 (0, 0, 0.5)（往前走），对极线变成什么样子？（提示：极点在图像中心）
2. 在 `triangulate` 的输入上加 0.001 的噪声，重算 3D 点的误差和深度的关系。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 两窗看一树，对应点落在对极线上，x₂ᵀ E x₁ 为零；
> 像素坐标用 F，K 逆转置夹着 E；
> 平面或纯转是单应，四对点就够；
> 三角化求点、PnP 求位姿；
> RANSAC 次数 log(1−p) 除 log(1−wˢ)；DUSt3R 直接回归点图。